# Attention Heatmaps for Lecture Slides

This notebook generates publication-ready heatmaps that explain which tokens attend to which others. We style the plots for inclusion in lecture decks or papers.


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

sns.set_theme(style="white", font_scale=1.0)
MODEL_NAME = "sshleifer/tiny-gpt2"


## Configure prompts

Select text pairs that highlight long-range dependencies. Monetary policy statements, risk reports, or DSGE summaries work well.


In [ ]:
prompts = [
    "When inflation expectations drift, central banks adjust guidance to restore anchors.",
    "Macroprudential tightening moderates credit booms yet preserves growth momentum.",
]


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, output_attentions=True)


def attention_heatmap(prompt: str, layer: int = -1, save_path: str | None = None):
    inputs = tokenizer(prompt, return_tensors="pt")
    with torch.no_grad():
        outputs = model(**inputs)
    attn = outputs.attentions[layer][0].mean(dim=0).numpy()
    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

    fig, ax = plt.subplots(figsize=(10, 6))
    sns.heatmap(attn, xticklabels=tokens, yticklabels=tokens, cmap="rocket", ax=ax,
                cbar_kws={"label": "Weight"})
    ax.set_title(f"Layer {layer % len(outputs.attentions)} attention weights")
    plt.xticks(rotation=70)
    plt.yticks(rotation=0)
    plt.tight_layout()

    if save_path:
        fig.savefig(save_path, dpi=300)
        print(f"Saved figure to {save_path}")

    return attn


In [ ]:
for i, prompt in enumerate(prompts):
    print(prompt)
    attention_heatmap(prompt, layer=-1, save_path=f"../docs/diagrams/attention_heatmap_{i+1}.png")


## Econometric takeaways

- Heatmaps look and feel like covariance or correlation matrices, lowering the barrier for students.
- Comparing prompts is analogous to running the same specification on different samples – divergences highlight structural breaks in attention patterns.
- Saving figures lets you insert them into `docs/lecture_slides` with zero extra processing.
